# Visual-language assistant with MiniCPM-V 4.7 and OpenVINO

MiniCPM-V 4.7 combines a SigLIP vision encoder with a Qwen3.5 language model. Compared with [MiniCPM-V 4.6](../minicpm-v-4.6/minicpm-v-4.6.ipynb), it uses **canvas M-RoPE** to preserve the spatial relationship of image slices and the temporal order of video frames. See the [Transformers documentation](https://huggingface.co/docs/transformers/main/model_doc/minicpmv4_7).

This tutorial exports either the **1B dense** or **35B-A3B MoE** checkpoint using [Optimum Intel](https://github.com/huggingface/optimum-intel/pull/2033), runs inference with OpenVINO, and demonstrates text, single-image, multi-image, document, and video inputs. The 35B-A3B checkpoint requires substantially more memory; its GPU PagedAttention INT4 path also depends on [this OpenVINO fix](https://github.com/openvinotoolkit/openvino/pull/38631).

Select a checkpoint and provide either a local directory or a Hugging Face model ID. This notebook uses the Optimum Intel `OVModelForVisualCausalLM` path, **not** the separate [GenAI](https://github.com/openvinotoolkit/openvino.genai/pull/4591) `VLMPipeline` API. The OpenVINO runtime in the linked Optimum Intel PR supports **16x downsampling only**, even though the Transformers model also offers 4x.

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Convert and optimize the model](#Convert-and-optimize-the-model)
- [Prepare the OpenVINO pipeline](#Prepare-the-OpenVINO-pipeline)
- [Text and single-image inference](#Text-and-single-image-inference)
- [Multiple images and document text](#Multiple-images-and-document-text)
- [Video inference](#Video-inference)
- [Interactive image demo](#Interactive-image-demo)

⚠️ **EXPERIMENTAL NOTEBOOK**

This notebook demonstrates a model that has not been fully validated with OpenVINO and is using a custom branch of optimum-intel. It may be fully supported and validated in the future.

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/minicpm-v-4.7/minicpm-v-4.7.ipynb" />

### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

Exporting a model requires enough disk space and RAM for its original weights and OpenVINO representation. Run cells in order.

## Prerequisites
[back to top ⬆️](#Table-of-contents:)

Install Optimum Intel and Transformers in the order shown in the [4.7 export PR](https://github.com/huggingface/optimum-intel/pull/2033). Restart the kernel after installing if an older Transformers or OpenVINO was already imported.

In [ ]:
%pip install "git+https://github.com/rkazants/optimum-intel.git@minicpmv47" --extra-index-url https://download.pytorch.org/whl/cpu
%pip install transformers==5.18.0 --extra-index-url https://download.pytorch.org/whl/cpu
%pip install -q av Pillow requests ipywidgets "gradio>=6.29.1,<7" nncf
%pip install -q --pre -U openvino openvino-tokenizers --extra-index-url https://storage.openvinotoolkit.org/simple/wheels/nightly

In [ ]:
from pathlib import Path
import requests

for name in ("cmd_helper.py", "notebook_utils.py"):
    if not Path(name).exists():
        response = requests.get(
            f"https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/{name}",
            timeout=30,
        )
        response.raise_for_status()
        Path(name).write_text(response.text, encoding="utf-8")

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("minicpm-v-4.7.ipynb")

## Convert and optimize the model
[back to top ⬆️](#Table-of-contents:)

Choose **MiniCPM-V-4.7-1B** or **MiniCPM-V-4.7-35B-A3B** below. An empty `Model:` field uses a local directory with the selected checkpoint's name. Otherwise, enter the corresponding local directory or a Hugging Face ID in `org/repo` form. Keep the selected checkpoint and source consistent.

In [ ]:
import ipywidgets as widgets

checkpoint = widgets.Dropdown(
    options=["MiniCPM-V-4.7-1B", "MiniCPM-V-4.7-35B-A3B"],
    value="MiniCPM-V-4.7-1B",
    description="Checkpoint:",
)
model_source = widgets.Text(description="Model:", placeholder="Local path or org/repo (optional)")
weight_format = widgets.Dropdown(options=["int8", "int4", "fp16"], value="int8", description="Weights:")
display(checkpoint, model_source, weight_format)

In [ ]:
from cmd_helper import optimum_cli
import os
import sys

model_id = model_source.value.strip() or checkpoint.value
if not Path(model_id).is_dir() and "/" not in model_id:
    raise FileNotFoundError(f"Provide a local checkpoint directory or a Hugging Face model ID (org/repo): {model_id}")

model_dir = Path("MiniCPM-V-4.7-ov") / checkpoint.value / Path(model_id).name / weight_format.value
if not model_dir.exists():
    os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ["PATH"]
    optimum_cli(
        model_id,
        model_dir,
        additional_args={"task": "image-text-to-text", "weight-format": weight_format.value},
    )
print(f"OpenVINO model: {model_dir}")

## Prepare the OpenVINO pipeline
[back to top ⬆️](#Table-of-contents:)

Optimum Intel exports vision embeddings, text embeddings, and the language model. Its 4.7 branch assembles the native MiniCPM-V 4.7 processor from the 4.6 image/video processors when needed, then saves it with the exported model. We use the exported processor and the Optimum Intel runtime, which supplies the new canvas M-RoPE positions. Keep the processor and exported weights from the same checkpoint.

In [ ]:
from notebook_utils import device_widget

device = device_widget(default="CPU", exclude=["NPU"])
device

In [ ]:
from gradio_helper import load_model

processor, ov_model = load_model(str(model_dir.resolve()), device.value)
print(f"Loaded {model_dir} on {device.value} (16x downsampling)")

## Text and single-image inference
[back to top ⬆️](#Table-of-contents:)

Use the chat template for every example; the model expects its multimodal markers and thinking-mode prompt format. `ask(...)` streams tokens to the cell and returns the full response. All examples keep the **16x default** and disable thinking for short, comparable outputs. Pass `enable_thinking=True` to `ask(...)` for a longer thinking-mode response, for example `ask(cat_messages, max_new_tokens=1024, enable_thinking=True)`.

In [ ]:
from io import BytesIO
from PIL import Image
from gradio_helper import stream_generate


def load_image(url):
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    with Image.open(BytesIO(response.content)) as image:
        return image.convert("RGB")


def ask(messages, max_new_tokens=128, *, enable_thinking=False, **template_kwargs):
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        enable_thinking=enable_thinking,
        **template_kwargs,
    )
    chunks = []
    for chunk in stream_generate(processor, ov_model, inputs, max_new_tokens=max_new_tokens):
        print(chunk, end="", flush=True)
        chunks.append(chunk)
    print()
    return "".join(chunks)


text_messages = [{"role": "user", "content": [{"type": "text", "text": "What is the capital of France? Answer in one word."}]}]
print("Text only:", end=" ", flush=True)
_ = ask(text_messages)

In [ ]:
cat = load_image("https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg")
display(cat)
cat_messages = [{"role": "user", "content": [{"type": "image", "image": cat}, {"type": "text", "text": "Describe this image."}]}]
_ = ask(cat_messages)

## Multiple images and document text
[back to top ⬆️](#Table-of-contents:)

Multiple separately addressable images can use `use_image_id`. A document with small text exercises image slicing and the spatial arrangement of its patches; the synthetic invoice below needs no external demo asset. Its answer is illustrative, not a quality guarantee.

In [ ]:
bee = load_image("https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/bee.jpg")
multi_messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": cat},
            {"type": "image", "image": bee},
            {"type": "text", "text": "What is shown in the first image, and what is shown in the second?"},
        ],
    }
]
_ = ask(multi_messages, processor_kwargs={"use_image_id": True})

In [ ]:
from PIL import ImageDraw

invoice = Image.new("RGB", (420, 220), "white")
draw = ImageDraw.Draw(invoice)
draw.text((20, 30), "INVOICE # OV-4721", fill="black")
draw.text((20, 80), "DUE DATE: 2026-11-15", fill="black")
draw.text((20, 130), "TOTAL: 125 USD", fill="black")
invoice = invoice.resize((1680, 880), Image.Resampling.NEAREST)
display(invoice)
document_messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": invoice},
            {"type": "text", "text": "What are the invoice number and due date?"},
        ],
    }
]
_ = ask(document_messages)

## Video inference
[back to top ⬆️](#Table-of-contents:)

A video is **one temporal sequence**, not a set of individually numbered images. Sample up to eight frames for a manageable CPU example. PyAV avoids a system TorchCodec/FFmpeg dependency; `do_sample_frames=False` prevents the processor from resampling frames that have already been selected. Ask about event order to exercise temporal understanding.

In [ ]:
import av
import numpy as np
from notebook_utils import download_file

video_path = Path("football.mp4")
if not video_path.exists():
    download_file(
        "https://huggingface.co/datasets/openbmb/DemoCase/resolve/main/football.mp4",
        filename=video_path.name,
    )

frames = []
with av.open(str(video_path)) as container:
    fps = float(container.streams.video[0].average_rate or 1)
    next_second = 0
    for index, frame in enumerate(container.decode(video=0)):
        second = frame.time if frame.time is not None else index / fps
        if second < next_second:
            continue
        frames.append(frame.to_ndarray(format="rgb24"))
        next_second += 1
        if len(frames) == 8:
            break
if not frames:
    raise ValueError(f"No video frames decoded from {video_path}")
video_frames = np.stack(frames)
print(f"Selected {len(frames)} frames")

video_messages = [
    {
        "role": "user",
        "content": [
            {"type": "video", "video": video_frames},
            {"type": "text", "text": "What happens in this video, in chronological order?"},
        ],
    }
]
_ = ask(video_messages, max_new_tokens=256, do_sample_frames=False)

## Interactive image demo
[back to top ⬆️](#Table-of-contents:)

After exporting once, run the next cell alone—even after restarting the kernel. Keep `gradio_helper.py` beside the notebook; select a model/device and click an example row (preview, question, thinking). Responses stream, and thinking appears in a collapsible block. The model loads on the first request; if localhost is unavailable, Gradio creates a public `share=True` link.

In [ ]:
from gradio_helper import make_demo

demo = make_demo()
try:
    demo.launch(debug=True)
except ValueError as exc:
    if not str(exc).startswith("When localhost is not accessible, a shareable link must be created."):
        raise
    demo.launch(debug=True, share=True)
# If you are launching remotely, specify server_name and server_port
# EXAMPLE: `demo.launch(server_name='your server name', server_port='server port in int')`
# To learn more please refer to the Gradio docs: https://gradio.app/docs/